# Lab 8 — Delta Lake

## 1. Setup

This notebook runs on its own: the cells below recreate the required state from the first part.

In [1]:
import os, json, csv, random, shutil
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (
    SparkSession.builder
    .appName("Lab 8 - ETL Medallion + Delta")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()   # downloads the Delta jars the first time
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:31:35 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:31:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/felipe/anaconda3/envs/pyspark/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/felipe/.ivy2.5.2/cache
The jars for the packages stored in: /home/felipe/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1bcd69a6-1561-4ce8-ae0f-b66208ced29a;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 77ms :: artifacts dl 3ms
	:: module

Spark 4.0.1


In [2]:
LAKE       = "/tmp/lake"
RAW        = f"{LAKE}/raw"
BRONZE     = f"{LAKE}/bronze/orders"
SILVER     = f"{LAKE}/silver/orders"
QUARANTINE = f"{LAKE}/quarantine/orders"
GOLD       = f"{LAKE}/gold"

def tree(path):
    """Shows the folder structure with file counts and sizes."""
    for root, dirs, files in sorted(os.walk(path)):
        dirs.sort()
        data = [f for f in files if not f.startswith((".", "_"))]
        depth = root[len(path):].count(os.sep)
        size = sum(os.path.getsize(os.path.join(root, f)) for f in data) / 1024
        info = f"  ({len(data)} files, {size:,.0f} KB)" if data else ""
        print("    " * depth + (os.path.basename(root) or root) + "/" + info)

def scan_info(df):
    """Shows the filters and columns that reach the file read."""
    import io, contextlib, re
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    scan = next(l for l in buf.getvalue().splitlines() if "FileScan" in l)
    for key in ["PartitionFilters", "PushedFilters", "ReadSchema"]:
        m = re.search(key + r": (\[.*?\]|struct<.*?>)", scan)
        print(f"{key:<17}: {m.group(1) if m else '-'}")

In [3]:
REGIONS  = ["North", "Northeast", "Midwest", "Southeast", "South"]
STATUSES = ["created", "paid", "shipped", "delivered", "cancelled"]
PRODUCTS = [
    (1, "Notebook", "Computers", 3500), (2, "Monitor", "Computers", 1200),
    (3, "Mouse", "Peripherals", 85),    (4, "Keyboard", "Peripherals", 150),
    (5, "Webcam", "Peripherals", 320),  (6, "USB Hub", "Peripherals", 120),
    (7, "Headset", "Audio", 250),       (8, "Speaker", "Audio", 400),
    (9, "External HD", "Storage", 400), (10, "Pendrive", "Storage", 60),
    (11, "USB Cable", "Cables", 25),    (12, "HDMI Cable", "Cables", 40),
]

rng = random.Random(42)
next_id = [100_000]
known_orders = {}   # order_id -> created_at (to generate updates on later days)

def dirty_region(r):
    if rng.random() < 0.01: return "N/A"           # region not provided
    return rng.choice([r, r, r, r.lower(), " " + r.upper(), r + "  "])

def dirty_price(p):
    roll = rng.random()
    if roll < 0.03: return "N/A"
    if roll < 0.05: return f"-{p:.2f}"
    if roll < 0.25: return f"R$ {p:.2f}"
    return f"{p:.2f}"

def dirty_quantity():
    roll = rng.random()
    if roll < 0.02: return ""
    if roll < 0.04: return "0"
    return str(rng.randint(1, 4))

def make_order(order_id, created, updated, status, fmt):
    pid, _, _, base = rng.choice(PRODUCTS)
    if rng.random() < 0.02:
        pid = 999                                   # nonexistent product
    ts_created = created.strftime(fmt) if rng.random() > 0.01 else "2024-13-45 99:99"   # corrupted date
    return {
        "order_id":    str(order_id),
        "customer_id": str(rng.randint(1, 3000)) if rng.random() > 0.01 else "",
        "product_id":  str(pid),
        "quantity":    dirty_quantity(),
        "price":       dirty_price(base * rng.uniform(0.9, 1.1)),
        "region":      dirty_region(rng.choice(REGIONS)),
        "status":      status if rng.random() > 0.01 else "pending_review",       # status outside the domain
        "created_at":  ts_created,
        "updated_at":  updated.strftime(fmt),
    }

def generate_day(day, n_new=1200, n_updates=300):
    """Generates one day's files: new orders + updates to older orders."""
    rows_csv, rows_json = [], []
    for _ in range(n_new):
        oid = next_id[0]; next_id[0] += 1
        created = day + timedelta(minutes=rng.randint(0, 1439))
        is_app = rng.random() < 0.6
        fmt = "%Y-%m-%dT%H:%M:%S" if is_app else rng.choice(["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"])
        row = make_order(oid, created, created, rng.choice(["created", "paid"]), fmt)
        known_orders[oid] = (created, is_app, row)
        (rows_json if is_app else rows_csv).append(row)

    # status updates for orders from previous days
    old = [o for o, (c, _, _) in known_orders.items() if c.date() < day.date()]
    for oid in rng.sample(old, min(n_updates, len(old))):
        created, is_app, original = known_orders[oid]
        fmt = "%Y-%m-%dT%H:%M:%S" if is_app else "%Y-%m-%d %H:%M:%S"
        row = dict(original)
        row["status"] = rng.choice(["shipped", "delivered", "cancelled"])
        row["created_at"] = created.strftime(fmt)
        row["updated_at"] = (day + timedelta(minutes=rng.randint(0, 1439))).strftime(fmt)
        (rows_json if is_app else rows_csv).append(row)

    # exact duplicates (file resent, API retry...)
    rows_csv  += rng.sample(rows_csv,  len(rows_csv)  // 30)
    rows_json += rng.sample(rows_json, len(rows_json) // 30)

    tag = day.strftime("%Y%m%d")
    with open(f"{RAW}/orders_csv/pos_{tag}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows_csv[0].keys()))
        w.writeheader(); w.writerows(rows_csv)
    with open(f"{RAW}/orders_json/app_{tag}.json", "w") as f:
        for r in rows_json:
            r = dict(r, device=rng.choice(["ios", "android", "web"]))
            if rng.random() < 0.5 and r["price"].replace(".", "").isdigit():
                r["price"] = float(r["price"])      # in JSON, sometimes a number, sometimes text
            f.write(json.dumps(r) + "\n")
    return len(rows_csv), len(rows_json)

shutil.rmtree(LAKE, ignore_errors=True)
os.makedirs(f"{RAW}/orders_csv"); os.makedirs(f"{RAW}/orders_json")

with open(f"{RAW}/products.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["product_id", "product_name", "category", "list_price"]); w.writerows(PRODUCTS)

for d in ["2024-07-01", "2024-07-02", "2024-07-03"]:
    n_csv, n_json = generate_day(datetime.fromisoformat(d))
    print(f"{d}: {n_csv} CSV rows, {n_json} JSON rows")

tree(RAW)

2024-07-01: 505 CSV rows, 734 JSON rows
2024-07-02: 587 CSV rows, 962 JSON rows
2024-07-03: 584 CSV rows, 965 JSON rows
raw/  (1 files, 0 KB)
    orders_csv/  (3 files, 126 KB)
    orders_json/  (3 files, 608 KB)


In [4]:
raw_csv = (
    spark.read
    .option("header", True)                 # no inferSchema: everything is a string
    .csv(f"{RAW}/orders_csv")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("channel", F.lit("store"))
)

raw_json = (
    spark.read
    .option("primitivesAsString", True)     # numbers and booleans also become strings
    .json(f"{RAW}/orders_json")
    .select("*", F.col("_metadata.file_path").alias("_source_file"))
    .withColumn("channel", F.lit("app"))
)

bronze = (
    raw_csv.unionByName(raw_json, allowMissingColumns=True)
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze.write.mode("overwrite").parquet(BRONZE)
bronze = spark.read.parquet(BRONZE)

bronze.printSchema()
bronze.groupBy("channel").count().show()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- price: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- device: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

+-------+-----+
|channel|count|
+-------+-----+
|    app| 2661|
|  store| 1676|
+-------+-----+



In [5]:
TS_FORMATS = ["yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm", "yyyy-MM-dd'T'HH:mm:ss"]

def parse_ts(col):
    """Tries each format and keeps the first one that works."""
    return F.coalesce(*[F.try_to_timestamp(F.col(col), F.lit(fmt)) for fmt in TS_FORMATS])

def clean(df):
    return (
        df
        .withColumn("order_id",    F.col("order_id").try_cast("long"))
        .withColumn("customer_id", F.col("customer_id").try_cast("long"))
        .withColumn("product_id",  F.col("product_id").try_cast("int"))
        .withColumn("quantity",    F.col("quantity").try_cast("int"))
        .withColumn("price",       F.regexp_replace("price", r"R\$|\s", "").try_cast("double"))
        .withColumn("region",      F.initcap(F.trim("region")))
        .withColumn("status",      F.lower(F.trim("status")))
        .withColumn("created_at",  parse_ts("created_at"))
        .withColumn("updated_at",  parse_ts("updated_at"))
    )

products = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW}/products.csv")

def enrich(df):
    return (
        df.join(F.broadcast(products.select("product_id", "product_name", "category")), "product_id", "left")
        .withColumn("revenue", F.round(F.col("price") * F.col("quantity"), 2))
        .withColumn("order_date", F.to_date("created_at"))
    )

clean(bronze).select("order_id", "price", "quantity", "region", "created_at").show(5)

+--------+-------+--------+---------+-------------------+
|order_id|  price|quantity|   region|         created_at|
+--------+-------+--------+---------+-------------------+
|  102400| 260.16|       4|Southeast|2024-07-03 23:37:00|
|  102402|3314.45|       1|Northeast|2024-07-03 04:19:00|
|  102404| 251.04|       4|Southeast|2024-07-03 01:19:00|
|  102405| 267.12|       3|Northeast|2024-07-03 11:20:00|
|  102406| 439.52|       1|Southeast|2024-07-03 10:15:00|
+--------+-------+--------+---------+-------------------+
only showing top 5 rows


In [6]:
RULES = {
    "null_order_id":        F.col("order_id").isNull(),
    "invalid_price":        F.col("price").isNull() | (F.col("price") <= 0),
    "invalid_quantity":     F.col("quantity").isNull() | (F.col("quantity") <= 0),
    "invalid_date":         F.col("created_at").isNull() | F.col("updated_at").isNull(),
    "unknown_region":       F.col("region").isNull() | ~F.col("region").isin(REGIONS),
    "unknown_product":      F.col("category").isNull(),
}

def validate(df):
    errors = F.array_compact(F.array(*[F.when(cond, F.lit(name)) for name, cond in RULES.items()]))
    df = df.withColumn("_errors", errors)
    valid    = df.filter(F.size("_errors") == 0).drop("_errors")
    rejected = df.filter(F.size("_errors") > 0)
    return valid, rejected

def deduplicate(df):
    """Keeps only the latest version of each order."""
    w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc(), F.col("_ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

SILVER_COLS = ["order_id", "customer_id", "product_id", "product_name", "category",
               "quantity", "price", "revenue", "region", "status", "channel", "device",
               "created_at", "updated_at", "order_date", "_source_file", "_ingested_at"]

valid, rejected = validate(enrich(clean(bronze)))
silver_df = deduplicate(valid).select(SILVER_COLS)

print(f"Bronze    : {bronze.count():>6,} rows")
print(f"Valid     : {valid.count():>6,} rows")
print(f"Rejected  : {rejected.count():>6,} rows")
print(f"Silver    : {silver_df.count():>6,} unique orders")

Bronze    :  4,337 rows
Valid     :  3,809 rows
Rejected  :    528 rows
Silver    :  3,156 unique orders


---
## 2. The limits of plain Parquet

The pipeline works, but it has serious problems in production:

| Problem | Example |
|----------|---------|
| **No atomicity** | The job fails midway through an `overwrite`: the table is left half-written |
| **No UPDATE/MERGE** | A status update forces you to rewrite the partition (or the table) |
| **Inconsistent reads** | A reader sees old and new files mixed together during a write |
| **No history** | You cannot see what the table looked like yesterday |
| **Unprotected schema** | A file with different columns silently corrupts the table |

**Delta Lake** solves this by adding a **transaction log** next to the Parquet files.

```
  silver_delta/
  ├── _delta_log/
  │   ├── 00000000000000000000.json   <- version 0: "add files A, B, C"
  │   ├── 00000000000000000001.json   <- version 1: "remove B, add D"  (MERGE)
  │   └── 00000000000000000002.json   <- version 2: "remove A"         (DELETE)
  ├── order_date=2024-07-01/part-....parquet
  └── order_date=2024-07-02/part-....parquet
```

The data stays in Parquet. The log says **which files make up each version** of the table. Writing is atomic: either the log file for the new version exists, or the version did not happen.

---
## 3. Delta Lake in practice

### 3.1 Creating the Silver table in Delta

In [7]:
SILVER_DELTA = f"{LAKE}/silver_delta/orders"

(silver_df
 .repartition("order_date")
 .write.format("delta")
 .mode("overwrite")
 .partitionBy("order_date")
 .save(SILVER_DELTA))

orders_dt = DeltaTable.forPath(spark, SILVER_DELTA)
print("Rows:", orders_dt.toDF().count())
print("Log files:", sorted(os.listdir(f"{SILVER_DELTA}/_delta_log")))

Rows: 3156
Log files: ['.00000000000000000000.crc.crc', '.00000000000000000000.json.crc', '00000000000000000000.crc', '00000000000000000000.json', '_staged_commits']


In [8]:
# What is inside a commit
with open(f"{SILVER_DELTA}/_delta_log/00000000000000000000.json") as f:
    for line in f.readlines()[:4]:
        action = json.loads(line)
        print(list(action.keys())[0], "->", str(action)[:150])

commitInfo -> {'commitInfo': {'timestamp': 1793827902686, 'operation': 'WRITE', 'operationParameters': {'mode': 'Overwrite', 'partitionBy': '["order_date"]'}, 'isol
metaData -> {'metaData': {'id': '0afc15ee-1fbc-4b8c-8f91-bd7cd5118c2f', 'format': {'provider': 'parquet', 'options': {}}, 'schemaString': '{"type":"struct","field
protocol -> {'protocol': {'minReaderVersion': 1, 'minWriterVersion': 2}}
add -> {'add': {'path': 'order_date=2024-07-01/part-00000-80e06d67-efdd-40f4-8861-4928810fdabb.c000.snappy.parquet', 'partitionValues': {'order_date': '2024-


### 3.2 A new day arrived: MERGE (upsert)

On July 4, new orders and **updates to older orders** arrive. With Parquet, we would have to reprocess everything. With Delta we use `MERGE`:

```sql
MERGE INTO silver AS t
USING new_data AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *   -- only if newer
WHEN NOT MATCHED THEN INSERT *
```

The condition `s.updated_at > t.updated_at` protects against late data: an old event that arrives later does not overwrite a newer state.

In [9]:
n_csv, n_json = generate_day(datetime.fromisoformat("2024-07-04"))
print(f"New day: {n_csv} CSV rows, {n_json} JSON rows")

# Incremental Bronze: reads only the new files and appends
def read_raw(csv_glob, json_glob):
    c = (spark.read.option("header", True).csv(csv_glob)
         .select("*", F.col("_metadata.file_path").alias("_source_file")).withColumn("channel", F.lit("store")))
    j = (spark.read.option("primitivesAsString", True).json(json_glob)
         .select("*", F.col("_metadata.file_path").alias("_source_file")).withColumn("channel", F.lit("app")))
    return c.unionByName(j, allowMissingColumns=True).withColumn("_ingested_at", F.current_timestamp())

new_bronze = read_raw(f"{RAW}/orders_csv/pos_20240704.csv", f"{RAW}/orders_json/app_20240704.json")
new_bronze.write.mode("append").parquet(BRONZE)

# Silver: same functions as before
new_valid, new_rejected = validate(enrich(clean(new_bronze)))
updates = deduplicate(new_valid).select(SILVER_COLS)
new_rejected.write.mode("append").parquet(QUARANTINE)

print("Records for the MERGE:", updates.count())

New day: 642 CSV rows, 907 JSON rows
Records for the MERGE: 1336


In [10]:
# collect() now: the Delta DataFrame always reads the latest version when it runs
status_before = spark.createDataFrame(orders_dt.toDF().groupBy("status").count().collect())

(orders_dt.alias("t")
 .merge(updates.alias("s"), "t.order_id = s.order_id")
 .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
 .whenNotMatchedInsertAll()
 .execute())

status_after = orders_dt.toDF().groupBy("status").count()
(status_before.withColumnRenamed("count", "before")
 .join(status_after.withColumnRenamed("count", "after"), "status", "full")
 .orderBy("status").show())

# MERGE metrics are recorded in the history
(orders_dt.history(1)
 .select("version", "operation",
         F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"),
         F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"))
 .show())

+--------------+------+-----+
|        status|before|after|
+--------------+------+-----+
|     cancelled|   157|  230|
|       created|  1276| 1748|
|     delivered|   175|  265|
|          paid|  1349| 1706|
|pending_review|    31|   33|
|       shipped|   168|  240|
+--------------+------+-----+

+-------+---------+--------+-------+
|version|operation|inserted|updated|
+-------+---------+--------+-------+
|      1|    MERGE|    1066|    270|
+-------+---------+--------+-------+



### 3.3 History and time travel

Each operation creates a new version. Old versions remain readable until `VACUUM` removes them.

In [11]:
orders_dt.history().select("version", "timestamp", "operation").show(truncate=False)

v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_DELTA)
v1 = spark.read.format("delta").load(SILVER_DELTA)
print(f"Version 0: {v0.count():,} orders | Current version: {v1.count():,} orders")

# An order whose status changed between versions
changed = (v0.alias("a").join(v1.alias("b"), "order_id")
           .filter(F.col("a.status") != F.col("b.status"))
           .select("order_id", F.col("a.status").alias("status_v0"), F.col("b.status").alias("status_current")))
changed.show(5)

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|1      |2026-11-04 18:31:46.33 |MERGE    |
|0      |2026-11-04 18:31:42.758|WRITE    |
+-------+-----------------------+---------+

Version 0: 3,156 orders | Current version: 4,222 orders
+--------+---------+--------------+
|order_id|status_v0|status_current|
+--------+---------+--------------+
|  100027|  shipped|     cancelled|
|  100032|  created|     delivered|
|  100037|delivered|     cancelled|
|  100049|     paid|     cancelled|
|  100056|     paid|       shipped|
+--------+---------+--------------+
only showing top 5 rows


In [12]:
# The same in SQL
spark.sql(f"SELECT status, COUNT(*) AS n FROM delta.`{SILVER_DELTA}` VERSION AS OF 0 GROUP BY status").show()
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_DELTA}`").select("version", "operation").show()

+--------------+----+
|        status|   n|
+--------------+----+
|          paid|1349|
|     delivered| 175|
|       shipped| 168|
|pending_review|  31|
|       created|1276|
|     cancelled| 157|
+--------------+----+

+-------+---------+
|version|operation|
+-------+---------+
|      1|    MERGE|
|      0|    WRITE|
+-------+---------+



### 3.4 Schema enforcement and schema evolution

Delta **rejects** writes with an incompatible schema. This prevents a different file from corrupting the table. When the change is intentional, evolution is explicit with `mergeSchema`.

In [13]:
with_coupon = updates.limit(5).withColumn("coupon", F.lit("BLACKFRIDAY"))

try:
    with_coupon.write.format("delta").mode("append").save(SILVER_DELTA)
except Exception as e:
    print("Write rejected:", type(e).__name__)
    print(str(e).splitlines()[0][:200])

Write rejected: AnalysisException
[_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 0afc15ee-1fbc-4b8c-8f91-bd7cd5118c2f).


In [14]:
# Intentional evolution: the new column is added and old rows get null
(with_coupon.write.format("delta")
 .mode("append")
 .option("mergeSchema", "true")
 .save(SILVER_DELTA))

spark.read.format("delta").load(SILVER_DELTA).groupBy("coupon").count().show()

+-----------+-----+
|     coupon|count|
+-----------+-----+
|       NULL| 4222|
|BLACKFRIDAY|    5|
+-----------+-----+



> This 5-row append duplicated orders on purpose for the demo. We undo it in section 3.5.

### 3.5 DELETE, UPDATE and RESTORE

Typical example: a customer asks for their data to be deleted (GDPR, or LGPD in Brazil).

In [15]:
# Undoes the previous demo's append by going back to the MERGE version
merge_version = orders_dt.history().filter("operation = 'MERGE'").agg(F.max("version")).first()[0]
orders_dt.restoreToVersion(merge_version)
print("Restored to version", merge_version, "| rows:", orders_dt.toDF().count())

# GDPR/LGPD: delete all of a customer's orders
customer = orders_dt.toDF().select("customer_id").first()[0]
print(f"Orders of customer {customer} before:", orders_dt.toDF().filter(F.col("customer_id") == customer).count())
orders_dt.delete(F.col("customer_id") == customer)
print(f"Orders of customer {customer} after:", orders_dt.toDF().filter(F.col("customer_id") == customer).count())

# UPDATE: orders 'created' more than 2 days ago become 'expired'
orders_dt.update(
    condition="status = 'created' AND created_at < '2024-07-03'",
    set={"status": F.lit("expired")},
)
orders_dt.toDF().groupBy("status").count().orderBy("status").show()

Restored to version 1 | rows: 4222
Orders of customer 146 before: 1
Orders of customer 146 after: 0
+--------------+-----+
|        status|count|
+--------------+-----+
|     cancelled|  230|
|       created| 1059|
|     delivered|  265|
|       expired|  688|
|          paid| 1706|
|pending_review|   33|
|       shipped|  240|
+--------------+-----+



**Warning:** after `DELETE`, the customer's data still exists in the Parquet files of the old versions, and time travel can read it. For the deletion to be real, you must run `VACUUM`.

### 3.6 OPTIMIZE and VACUUM

| Command | What it does | When |
|---------|-----------|--------|
| `OPTIMIZE` | Compacts small files into larger ones | After many MERGEs/appends |
| `OPTIMIZE ... ZORDER BY (col)` | Also reorganizes data to skip more blocks in filters on `col` | Heavily filtered columns, outside the partition |
| `VACUUM` | Deletes from disk the files no recent version uses | Periodically; the default keeps 7 days |

In [16]:
print("Files before OPTIMIZE:", orders_dt.detail().first()["numFiles"])
orders_dt.optimize().executeZOrderBy("region")
print("Files after OPTIMIZE:", orders_dt.detail().first()["numFiles"])

def parquet_files(path):
    return sum(f.endswith(".parquet") for _, _, fs in os.walk(path) for f in fs)

print("\n.parquet files on disk before VACUUM:", parquet_files(SILVER_DELTA))

# For this lab only: zero retention. In production, keep the default (168 hours).
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
orders_dt.vacuum(0)
print(".parquet files on disk after VACUUM:", parquet_files(SILVER_DELTA))

try:
    n = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_DELTA).count()
    print("\nVersion 0 still readable:", n)
except Exception as e:
    print("\nTime travel to version 0 is no longer possible: the files were deleted.")

Files before OPTIMIZE: 4
Files after OPTIMIZE: 4

.parquet files on disk before VACUUM: 16
Deleted 11 files and directories in a total of 5 directories.
.parquet files on disk after VACUUM: 5

Version 0 still readable: 3156


### 3.7 Gold from the Delta Silver table

In [17]:
GOLD_DELTA = f"{LAKE}/gold_delta/daily_sales"

(spark.read.format("delta").load(SILVER_DELTA)
 .filter(~F.col("status").isin("cancelled", "expired"))
 .groupBy("order_date", "region")
 .agg(F.count("*").alias("orders"), F.round(F.sum("revenue"), 2).alias("revenue"))
 .write.format("delta").mode("overwrite").save(GOLD_DELTA))

spark.read.format("delta").load(GOLD_DELTA).orderBy("order_date", "region").show(10)

+----------+---------+------+---------+
|order_date|   region|orders|  revenue|
+----------+---------+------+---------+
|2024-07-01|  Midwest|   114|148939.25|
|2024-07-01|    North|   128|109341.57|
|2024-07-01|Northeast|   126|158188.36|
|2024-07-01|    South|   132|163602.39|
|2024-07-01|Southeast|   132|211311.33|
|2024-07-02|  Midwest|   125|206678.51|
|2024-07-02|    North|   110|145406.25|
|2024-07-02|Northeast|   117| 92072.44|
|2024-07-02|    South|   109|167902.29|
|2024-07-02|Southeast|   122|129511.75|
+----------+---------+------+---------+
only showing top 10 rows


---
## 4. Exercises

In [18]:
# ── Exercise 1 (Delta) ──────────────────────────────────────────────────────
# Simulate a LATE event: take an order already "shipped" or "delivered" and create a
# version of it with status "paid" and updated_at equal to created_at (that is, older).
# Run the MERGE and confirm it did NOT overwrite the current status.
# Use history() to show how many rows were updated.

# Your solution:
current = orders_dt.toDF().filter(F.col("status").isin("shipped", "delivered")).first()
print("Current state:", current["order_id"], current["status"], current["updated_at"])

late_event = (
    orders_dt.toDF().filter(F.col("order_id") == current["order_id"])
    .withColumn("status", F.lit("paid"))
    .withColumn("updated_at", F.col("created_at"))      # older than the current state
)

(orders_dt.alias("t")
 .merge(late_event.alias("s"), "t.order_id = s.order_id")
 .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
 .whenNotMatchedInsertAll()
 .execute())

print("After the MERGE:", orders_dt.toDF().filter(F.col("order_id") == current["order_id"]).first()["status"])
orders_dt.history(1).select("operation", F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated")).show()

Current state: 100004 delivered 2024-07-02 15:12:00
After the MERGE: delivered
+---------+-------+
|operation|updated|
+---------+-------+
|    MERGE|      0|
+---------+-------+



---
## Summary

| Layer | Format | Content | Typical operations |
|--------|---------|----------|-------------------|
| Bronze | Parquet/Delta, append | Raw data, all strings, + lineage | `read`, `unionByName`, `_metadata` |
| Silver | Delta, partitioned | Typed, validated, one row per entity | `try_cast`, rules, `row_number`, `MERGE` |
| Quarantine | Parquet/Delta | Rejected + reason | `array_compact`, `explode` |
| Gold | Delta | Business metrics | `groupBy`, window functions |

| Delta Lake | Command |
|------------|---------|
| Upsert | `DeltaTable.merge(...).whenMatchedUpdateAll().whenNotMatchedInsertAll()` |
| History | `.history()` / `DESCRIBE HISTORY` |
| Time travel | `.option("versionAsOf", n)` / `VERSION AS OF n` |
| Undo | `.restoreToVersion(n)` |
| Evolve schema | `.option("mergeSchema", "true")` |
| Compact | `.optimize().executeCompaction()` / `.executeZOrderBy(col)` |
| Clean up old versions | `.vacuum(hours)` |

In [19]:
spark.stop()